In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [20]:


def prepare_hospital_data(patients_path, services_path, schedule_path):
    # 1. Load Data
    patients_df = pd.read_csv(patients_path)
    services_df = pd.read_csv(services_path)
    schedule_df = pd.read_csv(schedule_path)
    
    # 2. Date Processing & Length of Stay 
    patients_df['arrival_date'] = pd.to_datetime(patients_df['arrival_date'])
    patients_df['departure_date'] = pd.to_datetime(patients_df['departure_date'])
    patients_df['length_of_stay'] = (patients_df['departure_date'] - patients_df['arrival_date']).dt.days
    
    # 3. Aggregate Patient Data to Weekly level
    min_date = patients_df['arrival_date'].min()
    patients_df['week'] = ((patients_df['arrival_date'] - min_date).dt.days // 7) + 1
    weekly_los = patients_df.groupby(['week', 'service'])['length_of_stay'].mean().reset_index()
    weekly_los.rename(columns={'length_of_stay': 'avg_los'}, inplace=True)
    
    # 4. Aggregate Staffing to Weekly level
    weekly_staff = schedule_df.groupby(['week', 'service'])['present'].sum().reset_index()
    weekly_staff.rename(columns={'present': 'staff_on_duty'}, inplace=True)
    
    # 5. The Master Merge (Fix: Use services_df, not s_df)
    df = pd.merge(services_df, weekly_staff, on=['week', 'service'], how='left')
    df = pd.merge(df, weekly_los, on=['week', 'service'], how='left')
    
    # 6. Target & Feature Engineering
    df['utilization_rate'] = (df['patients_admitted'] / df['available_beds']).clip(upper=1.0)
    df['demand_pressure'] = df['patients_request'] / df['available_beds']
    
    # 7. Time-Series Lags
    df = df.sort_values(['service', 'week'])
    df['utilization_lag1'] = df.groupby('service')['utilization_rate'].shift(1)
    
    # 8. Handling Nulls
    df['avg_los'] = df.groupby('service')['avg_los'].transform(lambda x: x.fillna(x.median()))
    df = df.dropna(subset=['utilization_lag1'])
    
    return df



In [21]:
# Create the actual dataframe
df = prepare_hospital_data(
    '../data/patients.csv', 
    '../data/services_weekly.csv', 
    '../data/staff_schedule.csv'
)

# Look at the shape - how many rows/cols did we end up with?
print(f"Master DF Shape: {df.shape}")
df.head()

Master DF Shape: (204, 15)


,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event,staff_on_duty,avg_los,utilization_rate,demand_pressure,utilization_lag1
7,2,1,ICU,16,7,7,0,79,85,donation,30,8.250000,0.437500,0.437500,1.000000
11,3,1,ICU,20,21,20,1,82,89,none,0,5.571429,1.000000,1.050000,0.437500
15,4,1,ICU,20,21,20,1,64,85,flu,29,8.250000,1.000000,1.050000,1.000000
19,5,2,ICU,22,13,13,0,73,88,none,28,10.666667,0.590909,0.590909,1.000000
23,6,2,ICU,14,12,12,0,87,51,none,0,7.500000,0.857143,0.857143,0.590909


In [22]:
# Check Week 1 vs Week 2 for the same service
df[['week', 'service', 'utilization_rate', 'utilization_lag1']].head(10)

,week,service,utilization_rate,utilization_lag1
7,2,ICU,0.437500,1.000000
11,3,ICU,1.000000,0.437500
15,4,ICU,1.000000,1.000000
19,5,ICU,0.590909,1.000000
23,6,ICU,0.857143,0.590909
27,7,ICU,1.000000,0.857143
31,8,ICU,0.833333,1.000000
35,9,ICU,1.000000,0.833333
39,10,ICU,0.615385,1.000000
43,11,ICU,1.000000,0.615385


In [23]:
df = pd.get_dummies(df, columns=['service', 'event'], drop_first=False)

## TRAIN/TEST SPLIT

TRAIN/TEST SPLIT. This won't be a random split, I will take the first 40 weeks of history to train the model on, and test on weeks 41-52 to see if model can forecast the future. 
Will drop leaky columns such as patients admitted, and post admission outcomes like patient satisfaction. 

In [28]:
TRAIN_CUTOFF = 40

DROP_COLS = [
    'utilization_rate',
    'patients_admitted',
    'patients_refused',
    'patient_satisfaction',
    'month',
]

feature_cols = [c for c in df.columns if c not in DROP_COLS]

train = df[df['week']<= TRAIN_CUTOFF]
test = df[df['week'] > TRAIN_CUTOFF]

X_train = train[feature_cols]
y_train = train['utilization_rate']
X_test = test[feature_cols]
y_test = test['utilization_rate']

print(f"Train: {len(train)} rows | Test: {len(test)} rows")
print(f"Features: {len(feature_cols)}")

Train: 156 rows | Test: 48 rows
Features: 16


## Model Training

For the model training, I will use gradient boosting because of its ability to handle tabular data with non-linear relationships. To optimize the model, I will utilise a time series split on the training data, this will help ensure to select the best hyperparameters. 

In [31]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'n_estimators': [100,200],
    'max_depth': [ 3,4,5],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid= param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation MAE: {-grid_search.best_score_:.4f}")

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best cross-validation MAE: 0.0059


In [32]:
final_model = grid_search.best_estimator_
final_preds = final_model.predict(X_test)
final_preds = np.clip(final_preds, 0, 1)
final_MAE = mean_absolute_error(y_test, final_preds)
print(F"Final Test Set MAE: {final_MAE:.4f}")


Final Test Set MAE: 0.0013
